In [182]:
import matplotlib.pyplot as plt
import robotic as ry
import numpy as np
import os
import re

In [ ]:
# loading, randomly placing and coloring the voxels. 
base = "../voxel_generation/only_structural_consistency/"
C = ry.Config()
ids = []
p = [0, 0, .05]
for i, file in enumerate(os.listdir(base)[:5]):
    ids.append(file.split('.')[0])
    C.addFile(base+file, namePrefix = file.split('.')[0] + "_").setPosition(p)

    if np.random.rand() > .6:
        p[0] += .2
    elif np.random.rand() > .3:
        p[1] += .2
    else:
        p[2] += .2

for id in ids:
    pattern = re.compile(rf"{id}_" + "cube*")
    filtered = [x for x in C.getFrameNames() if pattern.search(x)]

    c = np.random.rand(3)
    for cube in filtered:
        C.getFrame(cube).setColor(c)

In [ ]:
# calculating the maximum volume of the joing bounding box of all the voxels. This would help to align the camera
min_corner = np.array([np.inf, np.inf, np.inf])
max_corner = np.array([-np.inf, -np.inf, -np.inf])

for f in C.getFrames():
    size = f.getSize()
    if size is None or len(size) < 3:
        continue

    pos = np.array(f.getPosition())

    half = np.array(size[:3]) / 2

    min_corner = np.minimum(min_corner, pos - half)
    max_corner = np.maximum(max_corner, pos + half)

box_center = (max_corner + min_corner) / 2

In [ ]:
# getting the image
def get_img(C:ry.Config, camera_view:ry.CameraView=None, cam_f:str="cam0"):
    if camera_view is None:
        camera_view = ry.CameraView(C)
    cam = C.getFrame(cam_f)
    camera_view.setCamera(cam)
    img, depth = camera_view.computeImageAndDepth(C)
    img = np.asarray(img)
    return img, depth, camera_view

In [583]:
C.view()

37

In [ ]:
# aligning the camera to the center of the bounding box
C.addFrame("world").setPosition(box_center)

-- WARNING:kin.cpp:addFrame:204(-1) frame 'world' already exists! returning existing without modifications!


In [ ]:
# getting images from different views
C.delFrame("cam_dim_0")
C.delFrame("cam_dim_1")
C.delFrame("cam_dim_2")
C.delFrame("cam_dim_3")
C.delFrame("cam_dim_4")

cam = C.addFrame(name="cam_dim_0", parent="world", args='Q:"t(0 0 5) d(180 1 0 0)", shape:camera, width:500, height:500')
cam = C.addFrame(name="cam_dim_1", parent="world", args='Q:"t(0 4 .1) d(180 1 0 0) d(-90 1 0 0) d(180 0 0 1)", shape:camera, width:500, height:500')
cam = C.addFrame(name="cam_dim_2", parent="world", args='Q:"t(0 -4 .1) d(180 1 0 0) d(90 1 0 0)", shape:camera, width:500, height:500')
cam = C.addFrame(name="cam_dim_3", parent="world", args='Q:"t(4 0 .1) d(180 1 0 0) d(-90 1 0 0) d(180 0 0 1) d(90 0 1 0)", shape:camera, width:500, height:500')
cam = C.addFrame(name="cam_dim_4", parent="world", args='Q:"t(-4 0 .1) d(180 1 0 0) d(-90 1 0 0) d(180 0 0 1) d(-90 0 1 0)", shape:camera, width:500, height:500')

for i in range(5):
    t = get_img(C=C, cam_f = f"cam_dim_{i}")
    plt.imsave(f"image_dim_{i}.png", t[0])
    plt.imsave(f"depth_dim_{i}.png", t[1])

In [ ]:
# helps saving the environment
open("./highly_cluttered_env.g", "x").write(C.write())

4059